# Test Results and Visualizations

## Overview
This notebook performs a **visual and statistical evaluation** of the Machine Learning models trained in Notebook 02. Using **subject‑wise out‑of‑fold predictions**, we guarantee that no subject appears in both training and test sets, providing an honest estimate of generalization performance.

All generated figures are saved in the `processed/` directory and used in the final report.

## Pipeline Summary
1. **Setup & Data Loading**  
2. **Load Trained Models**  
3. **Generate Out‑of‑Fold Predictions** (subject‑wise)  
4. **Quantitative Metrics** (classification reports, summary table)  
5. **Confusion Matrices**  
6. **ROC‑AUC Curves**  
7. **Performance Comparison Charts**  
8. **Per‑Class F1 Heatmap**  
9. **Cross‑Check with Nested CV Results**  
10. **Final Summary**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import os

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc
)
from sklearn.multiclass import OneVsRestClassifier
from itertools import cycle

PROCESSED_DIR = '../data/processed/'
MODELS_DIR    = '../data/processed/models/'

sns.set_theme(style='whitegrid', palette='muted')
print('All imports successful.')

In [ ]:
# Load feature matrix
df = pd.read_csv(os.path.join(PROCESSED_DIR, 'eeg_rbp_features.csv'))

# Load metadata saved by Notebook 02
le           = joblib.load(os.path.join(MODELS_DIR, 'label_encoder.joblib'))
FEATURE_COLS = joblib.load(os.path.join(MODELS_DIR, 'feature_cols.joblib'))

METADATA_COLS = ['subject_id', 'epoch_id', 'label']
X      = df[FEATURE_COLS].values
groups = df['subject_id'].values
y      = le.transform(df['label'].values)

CLASS_NAMES = list(le.classes_)   # e.g. ['AD', 'CN', 'FTD']
N_CLASSES   = len(CLASS_NAMES)

print(f'Dataset shape : {X.shape}')
print(f'Classes       : {CLASS_NAMES}')
print(f'Unique subjects: {len(np.unique(groups))}')

In [ ]:
# Load all trained models from Notebook 02
MODEL_FILES = {
    'KNN':                 'knn.joblib',
    'Decision Tree':       'decision_tree.joblib',
    'SVM':                 'svm.joblib',
    'Logistic Regression': 'logistic_regression.joblib',
    'LDA':                 'lda.joblib',
}

MODELS = {}
for name, fname in MODEL_FILES.items():
    path = os.path.join(MODELS_DIR, fname)
    MODELS[name] = joblib.load(path)
    print(f'  Loaded: {name}')

print(f'\n✓ {len(MODELS)} models loaded.')

In [ ]:
# Generate subject-wise test predictions using GroupKFold
# This mirrors the exact split used in Notebook 02 to maintain integrity
gkf = GroupKFold(n_splits=5)

# Collect out-of-fold predictions for each model
oof_preds  = {name: np.zeros(len(y), dtype=int)         for name in MODELS}
oof_probas = {name: np.zeros((len(y), N_CLASSES))       for name in MODELS}

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train         = y[train_idx]

    for name, pipeline in MODELS.items():
        pipeline.fit(X_train, y_train)
        oof_preds[name][test_idx]  = pipeline.predict(X_test)
        oof_probas[name][test_idx] = pipeline.predict_proba(X_test)

    print(f'  Fold {fold+1}/5 complete')

print('\n✓ Out-of-fold predictions generated for all models.')

In [ ]:
# Comprehensive classification report for each model
for name in MODELS:
    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'{"="*55}')
    print(classification_report(y, oof_preds[name], target_names=CLASS_NAMES))

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Comparative summary table
rows = []
for name in MODELS:
    rows.append({
        'Model'    : name,
        'Accuracy' : accuracy_score(y, oof_preds[name]),
        'Precision': precision_score(y, oof_preds[name], average='macro', zero_division=0),
        'Recall'   : recall_score(y, oof_preds[name], average='macro', zero_division=0),
        'F1 Macro' : f1_score(y, oof_preds[name], average='macro', zero_division=0),
    })

summary_df = pd.DataFrame(rows).set_index('Model').sort_values('F1 Macro', ascending=False)

print('=== Comparative Summary Table ===')
display(summary_df.round(4))

best = summary_df['F1 Macro'].idxmax()
print(f'\n🏆 Best model by F1-Macro: {best} ({summary_df.loc[best, "F1 Macro"]:.4f})')

In [ ]:
n_models = len(MODELS)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 5))

for ax, name in zip(axes, MODELS):
    cm = confusion_matrix(y, oof_preds[name])
    
    # Normalize per true class (row) for easier reading
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    
    sns.heatmap(
        cm_norm, annot=cm, fmt='d',
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        cmap='Blues', vmin=0, vmax=1,
        linewidths=0.5, linecolor='gray',
        ax=ax, cbar=False
    )
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

fig.suptitle('Confusion Matrices — Normalized (counts annotated)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrices.png')

In [ ]:
# Binarize labels for One-vs-Rest ROC
y_bin = label_binarize(y, classes=list(range(N_CLASSES)))

colors = cycle(['#e63946', '#457b9d', '#2a9d8f', '#e9c46a', '#264653'])
line_styles = ['-', '--', '-.', ':', '-']

fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 5), sharey=True)

for ax, (name, color) in zip(axes, zip(MODELS, colors)):
    probas = oof_probas[name]
    
    auc_scores = []
    for i, (cls_name, ls) in enumerate(zip(CLASS_NAMES, line_styles)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], probas[:, i])
        roc_auc     = auc(fpr, tpr)
        auc_scores.append(roc_auc)
        ax.plot(fpr, tpr, lw=2, linestyle=ls,
                label=f'{cls_name} (AUC={roc_auc:.3f})')
    
    # Macro-average AUC
    mean_auc = np.mean(auc_scores)
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC=0.500)')
    ax.set_title(f'{name}\nMacro AUC={mean_auc:.3f}', fontsize=11, fontweight='bold')
    ax.set_xlabel('False Positive Rate')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('True Positive Rate')
fig.suptitle('ROC-AUC Curves — One-vs-Rest Multi-class',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'roc_auc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: roc_auc_curves.png')

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Macro']
x       = np.arange(len(summary_df))
width   = 0.2
palette = ['#457b9d', '#2a9d8f', '#e9c46a', '#e63946']

fig, ax = plt.subplots(figsize=(12, 5))

for i, (metric, color) in enumerate(zip(metrics, palette)):
    bars = ax.bar(x + i * width, summary_df[metric], width,
                  label=metric, color=color, alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{bar.get_height():.3f}',
                ha='center', va='bottom', fontsize=7)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(summary_df.index, fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison — All Metrics', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
ax.axhline(y=1/N_CLASSES, color='gray', linestyle='--', lw=1,
           label=f'Random baseline ({1/N_CLASSES:.2f})')
plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'performance_comparison.png'), dpi=150)
plt.show()
print('Saved: performance_comparison.png')

In [ ]:
# Build per-class F1 matrix (models × classes)
f1_matrix = []
for name in MODELS:
    report = classification_report(y, oof_preds[name],
                                   target_names=CLASS_NAMES,
                                   output_dict=True, zero_division=0)
    f1_matrix.append([report[cls]['f1-score'] for cls in CLASS_NAMES])

f1_df = pd.DataFrame(f1_matrix, index=list(MODELS.keys()), columns=CLASS_NAMES)

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(f1_df, annot=True, fmt='.3f', cmap='YlGnBu',
            vmin=0, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Per-Class F1-Score by Model', fontsize=13, fontweight='bold')
ax.set_xlabel('Class')
ax.set_ylabel('Model')
plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'f1_per_class_heatmap.png'), dpi=150)
plt.show()
print('Saved: f1_per_class_heatmap.png')

In [ ]:
best_model  = summary_df['F1 Macro'].idxmax()
best_f1     = summary_df.loc[best_model, 'F1 Macro']
best_acc    = summary_df.loc[best_model, 'Accuracy']

print('=' * 55)
print('  FINAL RESULTS SUMMARY')
print('=' * 55)
print(f'  Best model    : {best_model}')
print(f'  Accuracy      : {best_acc:.4f}')
print(f'  F1 Macro      : {best_f1:.4f}')
print('=' * 55)
print(f'\nFiles saved to: {PROCESSED_DIR}')
print('  - confusion_matrices.png')
print('  - roc_auc_curves.png')
print('  - performance_comparison.png')
print('  - f1_per_class_heatmap.png')
print('\n✓ Analysis complete.')